# S4.dev — attempt-1 generations on the 30 dev-plots (Kaggle T4)

**RUNNER ONLY.** Clones, installs, calls scripts. No logic here.

⛔ **NOT A RESULT.** These 120 generations are the substrate `w` and τ are
fitted on (`protocol.md` §S4 decisions 1, 2, 19). **No Critic runs** — it
requires both `w` and τ, and neither exists until this file does.

Generator: **`google/gemma-3-12b-it`**, single arm (registered 2026-08-12 —
the Bangla-adapted arm collapsed when TigerLLM's weights turned out to be
gemma's byte-for-byte; the comparison is scheduled as a robustness check).

**Settings: GPU T4 x2 · Internet ON · attach `bn-clean` dataset · attach
Models → google/gemma-3 → transformers/gemma-3-12b-it.**

Grid: 30 plots × 2 levels × 2 prompt arms = **120 generations**, ~35–50 min.


## 0. Model mount + repo


In [ ]:
import glob
hits = sorted(glob.glob('/kaggle/input/gemma-3/transformers/*12b-it*/*') +
              glob.glob('/kaggle/input/models/google/gemma-3/transformers/*12b-it*/*'))
assert hits, 'Add Input -> Models -> google/gemma-3 -> transformers/gemma-3-12b-it'
GEMMA_PATH = hits[-1]
print('GEMMA_PATH =', GEMMA_PATH)


In [ ]:
!git clone -q https://github.com/alphapie77/BSc_Thesis.git repo
%cd repo
!git log --oneline -1
!pip -q install chromadb sentence-transformers pyyaml bitsandbytes accelerate 2>&1 | tail -2


In [ ]:
!mkdir -p data/cleaned
!cp /kaggle/input/datasets/alphapie77/bn-clean/bn_clean.csv data/cleaned/bn_clean.csv || \
 cp /kaggle/input/bn-clean/bn_clean.csv data/cleaned/bn_clean.csv
!ls -l data/cleaned/


## 1. Index — expect `886` rows, digest `85fc2d7d…`

A different digest means different rows went in and nothing downstream is
comparable. Stop if it differs.


In [ ]:
!python src/agents/build_index.py --config configs/s4_index.yaml


## 2. Dry run — read the rendered prompt

Prints the full prompt for both levels and both prompt arms. Two of the
three bugs found on 2026-08-11 were caught here, not by the tests.


In [ ]:
!python src/agents/run_devplots.py --config configs/s4_devplots.yaml --dry-run 2>&1 | head -100


## 3. Generate

Resumable: each generation is appended to JSONL as it completes and a
re-run skips completed keys (the key carries `provider`, fixed 2026-08-12).


In [ ]:
# One line, no backslash continuation: IPython expands $VAR only on the FIRST
# line of a ! cell, so the continued form silently passed an EMPTY path and the
# run aborted at the grid (2026-08-15). Exporting to the environment first makes
# the expansion the shell's job, which does not depend on cell layout.
import os; os.environ['GEMMA_PATH'] = GEMMA_PATH
!python src/agents/run_devplots.py --config configs/s4_devplots.yaml --model-path arm_a=$GEMMA_PATH


## 4. Save out

⚠️ Kaggle wipes disk between sessions and these generations **cannot be
regenerated bit-for-bit on another host**. The JSONL is the artifact.
No `2>/dev/null`: a failed copy must be seen.


In [ ]:
!cp results/s4_devplot_generations.jsonl /kaggle/working/
!cp results/s4_devplot_generations.md results/s4_devplot_generations.json /kaggle/working/
!python src/common/env_snapshot.py && cp results/env_snapshot.json /kaggle/working/env_snapshot_s4dev_kaggle.json
!ls -lh /kaggle/working/
